---
**Script name:** 04_03_05_export_results.ipynb

**Purpose of script:** This script is to export the corrected data from the corrupted Nodes.

**Author:** Dr Katie Allan

**Date Created:** 2022-07-10

**Email:** katie.allan@sussex.ac.uk

**License:** 

---

**Notes:** Change 'path_to_folder' to appropriate folder paths. Change 'postgresql://user:@localhost/db_name' to PostgreSQL databse login credentials.

---

## Importing libraries that are required

In [1]:
import time
from datetime import timedelta
from scipy.special import rel_entr
from scipy.stats import entropy
import sqlalchemy as psql
import re
import pickle
import pandas as pd
import numpy as np
import os

## Define variables

In [ ]:
# set path to raw node data
dir_path = Path('path_to_folder')
input_dir = Path("../../04_01_data_wrangling/output")
output_dir = Path(".")

## Functions that are used throughout the script

In [ ]:
def edit_node_files(dictionary_files):
    new_data = {}

    #set to match own node IDs
    node_names = {
        "ExpressWay_2":'376adf',
        "BusinessPark_1":'32800c',
        "BusinessPark_2":'32800c',
        "BusinessPark_3":'32800c',
        "TeignBridge_1":'37920b',
        "TeignBridge_2":'37920b', 
        "TeignBridge_3":'37920b',
        "LawelldownWood_1":'37807e',
        "LawelldownWood_2":'37807e',
        "WinstowCottages_1":'3781a5',
        "WinstowCottages_2":'3781a5',
        "WinstowCottages_3":'3781a5',
        "Underpass_1":'376f37',
        "Underpass_2":'376f37',
        "Underpass_3":'376f37',
        "LawellHouse_1":'328787',
        "LawellHouse_2":'328787',
        "CowCave_1":'3292c8',
        "CowCave_2":'3292c8',
        "GappahCows_1":'32b6e0',
        "GappahCows_2":'32b6e0',
        "GappahCows_3":'32b6e0',
        "HighlandPark":'32800c',
        "ExpressWay_1":'376adf',
        "KilnWood":'377401',
        "BatHouse":'3795d0',
        "Olchard_B_1":'3289f3',
        "Olchard_B_2":'3289f3', 
        "Olchard_A_1":'3750b3',
        "Olchard_A_2":'3750b3',
        "Olchard_C_1":'37792f',
        "Olchard_C_2":'37792f',
        "Olchard_D_1":'376a2b',
        "Olchard_D_2":'376a2b',
        "Clay_Pitts_Way":'327d6b',
        "Little_Lyndridge_Farm":'3289f3',
        "Sandygate_M1":'37792f'
    }
                        
    for outer_key, inner_dict in dictionary_files.items():
        # Iterate over the inner dictionary
        for inner_key, df in inner_dict.items():
            #get the corresponding node id
            node = node_names.get(inner_key, None)
            if node is None:
                raise KeyError(f"No matching node_id found for key: {inner_key}")

            new_df = df.assign(
                method = 'kl_div',
                data_size = len(df),
                path = 'python',
                radio_id = 5,
                node_id = node,
                validated = 0,
                station_id = None
            )
            new_data[inner_key] = new_df

    return new_data

In [ ]:
#function to edit correct nodes into sql format
def edit_node_files(dictionary_files):
    new_data = {}

    #set to match own node IDs
    node_names = {
        "node_3750b3_beep_4_correct":'3750b3',
        "node_3750b3_beep_5_correct":'3750b3',
        "node_376a2b_beep_3_correct":'376a2b',
        "node_376a2b_beep_4_correct":'376a2b',
        "node_376adf_beep_3_correct":'376adf',
        "node_377401_beep_3_correct":'377401',
        "node_37792f_beep_3_correct":'37792f',
        "node_37792f_beep_4_correct":'37792f',
        "node_3795d0_beep_3_correct":'3795d0'
    }

    #set to match own node file paths
    file_path = {
        "node_3750b3_beep_4_correct": f'{dir_path}/3750b3 - Olchard A/beep_4.csv',
        "node_3750b3_beep_5_correct": f'{dir_path}/3750b3 - Olchard A/beep_5.csv',
        "node_376a2b_beep_3_correct": f'{dir_path}/376a2b - Olchard D/beep_3.csv',
        "node_376a2b_beep_4_correct": f'{dir_path}/376a2b - Olchard D/beep_4.csv',
        "node_376adf_beep_3_correct": f'{dir_path}/376adf - Devon Express Way Layby/beep_3.csv',
        "node_377401_beep_3_correct": f'{dir_path}/377401 - Kiln Wood/beep_3.csv',
        "node_37792f_beep_3_correct": f'{dir_path}/37792f - Olchard C/beep_3.csv',
        "node_37792f_beep_4_correct": f'{dir_path}/37792f - Olchard C/beep_4.csv',
        "node_3795d0_beep_3_correct": f'{dir_path}/3795d0 - Chudleigh Bat House/beep_3.csv'
    }
                        
    for key, df in dictionary_files.items():
        #get the corresponding node id
        node = node_names.get(key, None)
        file = file_path.get(key, None)
        if node is None:
            raise KeyError(f"No matching node_id found for key: {key}")

        new_df = df.assign(
            method = 'ctt',
            data_size = None,
            path = file,
            radio_id = 5,
            node_id = node,
            validated = 0,
            station_id = None
        )
        new_data[key] = new_df
        
    return new_data

## Load Data

In [2]:
#Read edited pickle files
# list the subdirectories
filelist = os.listdir()

fileload = {}

# Create loop to import files
for file in filelist:
    
    #check folder matches node_id value
    if re.match(re.compile(r'.*corrected_data\.pickle'), file):
        file_name = file[:-7]
        print(file_name)
        unpickled_data = pd.read_pickle(file)
        fileload[file_name] = unpickled_data

bathouse_corrected_data
highlandpark_corrected_data
kilnwood_corrected_data
olchard_b_corrected_data
olchard_corrected_data
sandygate_corrected_data


In [ ]:
#Load correct parts of node files
with open(f'{input_dir}/correct_node_data.pickle', 'rb') as file:
        correct_dataframes = pickle.load(file)
with open(f'{output_dir}/highlandpark_correct_data.pickle', 'rb') as file:
        highlandpark_correct = pickle.load(file)

C:\Users\katie\AppData\Local\Temp\ipykernel_20832\447197879.py:3: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  correct_dataframes = pickle.load(file)


In [ ]:
#Load correct parts of node files
with open('"../../04_02_proof/04_02_02_simulation/simulation_error_estimate_results_summary_stats_absolute.pickle', 'rb') as file:
    simulation_summary = pickle.load(file)

## Data processing

In [3]:
#assign highland park to it's own dataframe
highlandpark_data = fileload['highlandpark_corrected_data']
highlandpark_data

,time,id,rssi
63994,2022-10-27 07:52:17,FF78AA66,-115
63995,2022-10-27 15:21:34,E1800000,-115
63996,2022-10-27 17:37:11,7FF80087,-115
63997,2022-10-27 17:55:55,6634617F,-115
63998,2022-10-27 17:57:36,78196666,-107
...,...,...,...
65768,2023-05-03 02:39:05,19E1FFD5,-117
65769,2023-05-03 05:29:55,78CB3307,-114
65770,2023-05-03 06:43:40,2A00B480,-116
65771,2023-05-03 06:54:35,78662DE1,-115


In [4]:
highlandpark_new = highlandpark_data.assign(
    method = 'kl_div',
    data_size = len(highlandpark_data),
    path = 'python',
    radio_id = 5,
    node_id = '32800c',
    validated = 0,
    station_id = None
)
highlandpark_new

,time,id,rssi,method,data_size,path,radio_id,node_id,validated,station_id
63994,2022-10-27 07:52:17,FF78AA66,-115,kl_div,1779,python,5,32800c,0,None
63995,2022-10-27 15:21:34,E1800000,-115,kl_div,1779,python,5,32800c,0,None
63996,2022-10-27 17:37:11,7FF80087,-115,kl_div,1779,python,5,32800c,0,None
63997,2022-10-27 17:55:55,6634617F,-115,kl_div,1779,python,5,32800c,0,None
63998,2022-10-27 17:57:36,78196666,-107,kl_div,1779,python,5,32800c,0,None
...,...,...,...,...,...,...,...,...,...,...
65768,2023-05-03 02:39:05,19E1FFD5,-117,kl_div,1779,python,5,32800c,0,None
65769,2023-05-03 05:29:55,78CB3307,-114,kl_div,1779,python,5,32800c,0,None
65770,2023-05-03 06:43:40,2A00B480,-116,kl_div,1779,python,5,32800c,0,None
65771,2023-05-03 06:54:35,78662DE1,-115,kl_div,1779,python,5,32800c,0,None


In [5]:
#remove from dictionary
del fileload['highlandpark_corrected_data']
fileload.keys()

dict_keys(['bathouse_corrected_data', 'kilnwood_corrected_data', 'olchard_b_corrected_data', 'olchard_corrected_data', 'sandygate_corrected_data'])

In [7]:
#edit all corrected files to match sql table
new_node_files = edit_node_files(fileload)
new_node_files

{'ExpressWay_2':                    time        id  rssi  method  data_size    path  radio_id  \
 0   2023-11-24 14:25:46  4B551E4B   -40  kl_div        315  python         5   
 1   2023-11-24 14:25:49  55331961   -33  kl_div        315  python         5   
 2   2023-11-24 14:25:56  4B551E4B   -37  kl_div        315  python         5   
 3   2023-11-24 14:25:59  55331961   -31  kl_div        315  python         5   
 4   2023-11-24 14:26:06  4B551E4B   -37  kl_div        315  python         5   
 ..                  ...       ...   ...     ...        ...     ...       ...   
 557 2024-01-22 14:41:30  55331961   -85  kl_div        315  python         5   
 558 2024-01-22 14:41:40  55331961   -79  kl_div        315  python         5   
 559 2024-01-22 14:41:51  55331961   -97  kl_div        315  python         5   
 560 2024-01-22 14:42:11  55331961  -104  kl_div        315  python         5   
 564 2024-01-22 16:26:01  55331961  -108  kl_div        315  python         5   
 
     node_

In [8]:
#combining altered dataframes into one
combined_new_node_files = pd.concat(new_node_files.values(), ignore_index=True)
combined_new_node_files

,time,id,rssi,method,data_size,path,radio_id,node_id,validated,station_id
0,2023-11-24 14:25:46,4B551E4B,-40,kl_div,315,python,5,376adf,0,None
1,2023-11-24 14:25:49,55331961,-33,kl_div,315,python,5,376adf,0,None
2,2023-11-24 14:25:56,4B551E4B,-37,kl_div,315,python,5,376adf,0,None
3,2023-11-24 14:25:59,55331961,-31,kl_div,315,python,5,376adf,0,None
4,2023-11-24 14:26:06,4B551E4B,-37,kl_div,315,python,5,376adf,0,None
...,...,...,...,...,...,...,...,...,...,...
145656,2023-11-01 11:32:04,55331961,-66,kl_div,350,python,5,37792f,0,None
145657,2023-11-01 11:32:14,55331961,-71,kl_div,350,python,5,37792f,0,None
145658,2023-11-01 11:32:24,55331961,-65,kl_div,350,python,5,37792f,0,None
145659,2023-11-01 11:32:34,55331961,-70,kl_div,350,python,5,37792f,0,None


In [10]:
#delete dataframes which have already been loaded via R
del correct_dataframes['node_328f57_beep_2_correct'], correct_dataframes['node_3750b3_beep_3_correct'], correct_dataframes['node_376adf_beep_4_correct'], correct_dataframes['node_3795d0_beep_2_correct'], correct_dataframes['node_379f1a_beep_3_correct'], correct_dataframes['node_3289f1_beep_1_correct']
correct_dataframes.keys()

dict_keys(['node_3750b3_beep_4_correct', 'node_3750b3_beep_5_correct', 'node_376a2b_beep_3_correct', 'node_376a2b_beep_4_correct', 'node_376adf_beep_3_correct', 'node_377401_beep_3_correct', 'node_37792f_beep_3_correct', 'node_37792f_beep_4_correct', 'node_3795d0_beep_3_correct'])

In [ ]:
#edit highlandpark data into sql format
highlandpark_correct_new = highlandpark_correct.assign(
    method = 'ctt',
    data_size = None,
    path = f'{dir_path}/32800c - Highland Park/beep_0.csv',
    radio_id = 5,
    node_id = '32800c',
    validated = 0,
    station_id = None
)

In [13]:
#edit all corrected files to match sql table
new_correct_dataframes = edit_node_files(correct_dataframes)
new_correct_dataframes

{'node_3750b3_beep_4_correct':                     time        id  rssi method data_size  \
 751  2023-11-06 17:11:52  662A6107   -97    ctt      None   
 752  2023-11-06 17:12:01  662A6107   -96    ctt      None   
 753  2023-11-06 17:12:11  662A6107   -78    ctt      None   
 754  2023-11-06 17:12:20  662A6107   -92    ctt      None   
 755  2023-11-06 17:12:29  662A6107   -94    ctt      None   
 ...                  ...       ...   ...    ...       ...   
 5099 2023-11-24 13:35:15  4B551E4B   -85    ctt      None   
 5100 2023-11-24 13:35:21  55331961   -90    ctt      None   
 5101 2023-11-24 13:35:25  4B551E4B   -85    ctt      None   
 5102 2023-11-24 13:35:31  55331961   -89    ctt      None   
 5103 2023-11-24 13:35:35  4B551E4B   -85    ctt      None   
 
                                                    path  radio_id node_id  \
 751   C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...         5  3750b3   
 752   C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...         5  

In [14]:
#combining altered dataframes into one
combined_new_correct_dataframes = pd.concat(new_correct_dataframes.values(), ignore_index=True)
combined_new_correct_dataframes

,time,id,rssi,method,data_size,path,radio_id,node_id,validated,station_id
0,2023-11-06 17:11:52,662A6107,-97,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,3750b3,0,None
1,2023-11-06 17:12:01,662A6107,-96,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,3750b3,0,None
2,2023-11-06 17:12:11,662A6107,-78,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,3750b3,0,None
3,2023-11-06 17:12:20,662A6107,-92,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,3750b3,0,None
4,2023-11-06 17:12:29,662A6107,-94,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,3750b3,0,None
...,...,...,...,...,...,...,...,...,...,...
21264,2024-01-06 19:04:58,2D344B2D,-97,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,3795d0,0,None
21265,2024-01-22 14:38:26,55331961,-107,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,3795d0,0,None
21266,2024-01-22 16:23:27,55331961,-111,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,3795d0,0,None
21267,2024-01-22 16:23:37,55331961,-111,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,3795d0,0,None


In [15]:
#join all node data into one dataframe to input into sql 
kl_divergence_data = pd.concat([combined_new_node_files, combined_new_correct_dataframes, highlandpark_new, highlandpark_correct_new])
kl_divergence_data

,time,id,rssi,method,data_size,path,radio_id,node_id,validated,station_id
0,2023-11-24 14:25:46,4B551E4B,-40,kl_div,315,python,5,376adf,0,None
1,2023-11-24 14:25:49,55331961,-33,kl_div,315,python,5,376adf,0,None
2,2023-11-24 14:25:56,4B551E4B,-37,kl_div,315,python,5,376adf,0,None
3,2023-11-24 14:25:59,55331961,-31,kl_div,315,python,5,376adf,0,None
4,2023-11-24 14:26:06,4B551E4B,-37,kl_div,315,python,5,376adf,0,None
...,...,...,...,...,...,...,...,...,...,...
67155,2023-05-09 13:21:57,55331961,-42,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None
67156,2023-05-09 13:22:07,55331961,-42,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None
67157,2023-05-09 13:22:16,55331961,-43,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None
67158,2023-05-09 13:22:26,55331961,-42,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None


In [16]:
#edit column names to match sql table
kl_divergence_data.rename(columns = {"id": "tag_id", "rssi": "tag_rssi"}, inplace=True)
kl_divergence_data

,time,tag_id,tag_rssi,method,data_size,path,radio_id,node_id,validated,station_id
0,2023-11-24 14:25:46,4B551E4B,-40,kl_div,315,python,5,376adf,0,None
1,2023-11-24 14:25:49,55331961,-33,kl_div,315,python,5,376adf,0,None
2,2023-11-24 14:25:56,4B551E4B,-37,kl_div,315,python,5,376adf,0,None
3,2023-11-24 14:25:59,55331961,-31,kl_div,315,python,5,376adf,0,None
4,2023-11-24 14:26:06,4B551E4B,-37,kl_div,315,python,5,376adf,0,None
...,...,...,...,...,...,...,...,...,...,...
67155,2023-05-09 13:21:57,55331961,-42,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None
67156,2023-05-09 13:22:07,55331961,-42,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None
67157,2023-05-09 13:22:16,55331961,-43,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None
67158,2023-05-09 13:22:26,55331961,-42,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None


In [17]:
#remove index from dataframe
kl_divergence_data = kl_divergence_data.reset_index(drop=True)

In [19]:
simulation_summary = simulation_summary[["sample_size", "50%", "max"]]
simulation_summary.rename(columns = {"50%": "median_error", "max": "max_error"}, inplace=True)
simulation_summary

,sample_size,median_error,max_error
0,100,0 days 00:43:22.500000,61 days 20:08:27
1,200,0 days 00:12:52.500000,61 days 20:25:07
2,300,0 days 00:04:46,34 days 07:35:47
3,400,0 days 00:03:32,25 days 16:05:07
4,500,0 days 00:05:15,60 days 23:46:47
5,600,0 days 00:03:32,37 days 16:25:07
6,700,0 days 00:03:32,37 days 16:25:07
7,800,0 days 00:02:42,38 days 20:14:17
8,900,0 days 00:03:06.500000,39 days 04:44:17
9,1000,0 days 00:03:22,39 days 04:44:17


In [20]:
#set conditions to join on depending on dataframe size
conditions = [
    (kl_divergence_data['data_size'] < 200),
    (kl_divergence_data['data_size'] >= 200) & (kl_divergence_data['data_size'] < 300),
    (kl_divergence_data['data_size'] >= 300) & (kl_divergence_data['data_size'] < 400),
    (kl_divergence_data['data_size'] >= 400) & (kl_divergence_data['data_size'] < 500),
    (kl_divergence_data['data_size'] >= 500) & (kl_divergence_data['data_size'] < 600),
    (kl_divergence_data['data_size'] >= 600) & (kl_divergence_data['data_size'] < 700),
    (kl_divergence_data['data_size'] >= 700) & (kl_divergence_data['data_size'] < 800),
    (kl_divergence_data['data_size'] >= 800) & (kl_divergence_data['data_size'] < 900),
    (kl_divergence_data['data_size'] >= 900) & (kl_divergence_data['data_size'] < 1000),
    (kl_divergence_data['data_size'] >= 1000) & (kl_divergence_data['data_size'] < 1100),
    (kl_divergence_data['data_size'] >= 1100) & (kl_divergence_data['data_size'] < 1200),
    (kl_divergence_data['data_size'] >= 1200) & (kl_divergence_data['data_size'] < 1300),
    (kl_divergence_data['data_size'] >= 1300) & (kl_divergence_data['data_size'] < 1400),
    (kl_divergence_data['data_size'] >= 1400) & (kl_divergence_data['data_size'] < 1500),
    (kl_divergence_data['data_size'] >= 1500) & (kl_divergence_data['data_size'] < 1600),
    (kl_divergence_data['data_size'] >= 1600) & (kl_divergence_data['data_size'] < 1700),
    (kl_divergence_data['data_size'] >= 1700) & (kl_divergence_data['data_size'] < 1800),
    (kl_divergence_data['data_size'] >= 1800) & (kl_divergence_data['data_size'] < 1900),
    (kl_divergence_data['data_size'] >= 1900) & (kl_divergence_data['data_size'] < 2000),
    (kl_divergence_data['data_size'] >= 2000) & (kl_divergence_data['data_size'] < 2100),
    (kl_divergence_data['data_size'] >= 2100) & (kl_divergence_data['data_size'] < 2200),
    (kl_divergence_data['data_size'] >= 2200) & (kl_divergence_data['data_size'] < 2300),
    (kl_divergence_data['data_size'] >= 2300) & (kl_divergence_data['data_size'] < 2400),
    (kl_divergence_data['data_size'] >= 2400) & (kl_divergence_data['data_size'] < 2500),
    (kl_divergence_data['data_size'] >= 2500) & (kl_divergence_data['data_size'] < 2600),
    (kl_divergence_data['data_size'] >= 2600) & (kl_divergence_data['data_size'] < 2700),
    (kl_divergence_data['data_size'] >= 2700) & (kl_divergence_data['data_size'] < 2800),
    (kl_divergence_data['data_size'] >= 2800) & (kl_divergence_data['data_size'] < 2900),
    (kl_divergence_data['data_size'] >= 2900) & (kl_divergence_data['data_size'] < 3000),
    (kl_divergence_data['data_size'] >= 3000)
]

results = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900, 2000, 2100, 2200, 2300, 2400, 2500, 2600, 2700, 2800, 2900, 3000]

#create merging column
kl_divergence_data['join_value'] = np.select(conditions, results)

In [21]:
#join error estimate of data together
kl_divergence_data = pd.merge(kl_divergence_data, simulation_summary, left_on = 'join_value', right_on = "sample_size", how='left')
kl_divergence_data

,time,tag_id,tag_rssi,method,data_size,path,radio_id,node_id,validated,station_id,join_value,sample_size,median_error,max_error
0,2023-11-24 14:25:46,4B551E4B,-40,kl_div,315,python,5,376adf,0,None,300,300.0,0 days 00:04:46,34 days 07:35:47
1,2023-11-24 14:25:49,55331961,-33,kl_div,315,python,5,376adf,0,None,300,300.0,0 days 00:04:46,34 days 07:35:47
2,2023-11-24 14:25:56,4B551E4B,-37,kl_div,315,python,5,376adf,0,None,300,300.0,0 days 00:04:46,34 days 07:35:47
3,2023-11-24 14:25:59,55331961,-31,kl_div,315,python,5,376adf,0,None,300,300.0,0 days 00:04:46,34 days 07:35:47
4,2023-11-24 14:26:06,4B551E4B,-37,kl_div,315,python,5,376adf,0,None,300,300.0,0 days 00:04:46,34 days 07:35:47
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234085,2023-05-09 13:21:57,55331961,-42,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None,0,NaN,NaT,NaT
234086,2023-05-09 13:22:07,55331961,-42,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None,0,NaN,NaT,NaT
234087,2023-05-09 13:22:16,55331961,-43,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None,0,NaN,NaT,NaT
234088,2023-05-09 13:22:26,55331961,-42,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None,0,NaN,NaT,NaT


In [22]:
#convert error to seconds
kl_divergence_data['median_error_seconds'] = kl_divergence_data['median_error'].dt.total_seconds()
kl_divergence_data['max_error_seconds'] = kl_divergence_data['max_error'].dt.total_seconds()
kl_divergence_data

,time,tag_id,tag_rssi,method,data_size,path,radio_id,node_id,validated,station_id,join_value,sample_size,median_error,max_error,median_error_seconds,max_error_seconds
0,2023-11-24 14:25:46,4B551E4B,-40,kl_div,315,python,5,376adf,0,None,300,300.0,0 days 00:04:46,34 days 07:35:47,286.0,2964947.0
1,2023-11-24 14:25:49,55331961,-33,kl_div,315,python,5,376adf,0,None,300,300.0,0 days 00:04:46,34 days 07:35:47,286.0,2964947.0
2,2023-11-24 14:25:56,4B551E4B,-37,kl_div,315,python,5,376adf,0,None,300,300.0,0 days 00:04:46,34 days 07:35:47,286.0,2964947.0
3,2023-11-24 14:25:59,55331961,-31,kl_div,315,python,5,376adf,0,None,300,300.0,0 days 00:04:46,34 days 07:35:47,286.0,2964947.0
4,2023-11-24 14:26:06,4B551E4B,-37,kl_div,315,python,5,376adf,0,None,300,300.0,0 days 00:04:46,34 days 07:35:47,286.0,2964947.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234085,2023-05-09 13:21:57,55331961,-42,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None,0,NaN,NaT,NaT,NaN,NaN
234086,2023-05-09 13:22:07,55331961,-42,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None,0,NaN,NaT,NaT,NaN,NaN
234087,2023-05-09 13:22:16,55331961,-43,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None,0,NaN,NaT,NaT,NaN,NaN
234088,2023-05-09 13:22:26,55331961,-42,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None,0,NaN,NaT,NaT,NaN,NaN


In [23]:
# Drop columns that aren't needed in dataframe
kl_divergence_data = kl_divergence_data.drop(['join_value', 'sample_size', 'median_error', 'max_error'], axis = 1)
kl_divergence_data

,time,tag_id,tag_rssi,method,data_size,path,radio_id,node_id,validated,station_id,median_error_seconds,max_error_seconds
0,2023-11-24 14:25:46,4B551E4B,-40,kl_div,315,python,5,376adf,0,None,286.0,2964947.0
1,2023-11-24 14:25:49,55331961,-33,kl_div,315,python,5,376adf,0,None,286.0,2964947.0
2,2023-11-24 14:25:56,4B551E4B,-37,kl_div,315,python,5,376adf,0,None,286.0,2964947.0
3,2023-11-24 14:25:59,55331961,-31,kl_div,315,python,5,376adf,0,None,286.0,2964947.0
4,2023-11-24 14:26:06,4B551E4B,-37,kl_div,315,python,5,376adf,0,None,286.0,2964947.0
...,...,...,...,...,...,...,...,...,...,...,...,...
234085,2023-05-09 13:21:57,55331961,-42,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None,NaN,NaN
234086,2023-05-09 13:22:07,55331961,-42,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None,NaN,NaN
234087,2023-05-09 13:22:16,55331961,-43,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None,NaN,NaN
234088,2023-05-09 13:22:26,55331961,-42,ctt,None,C:/Users/katie/OneDrive/Documents/Uni/PhD/CTT_...,5,32800c,0,None,NaN,NaN


In [24]:
# Define the start and end of the time range
start_time = pd.to_datetime('2023-03-26 01:00:00')
end_time = pd.to_datetime('2023-10-29 02:00:00')

# Apply the condition to add 1 hour only to times within the specified range
kl_divergence_data['time'] = kl_divergence_data['time'].apply(
    lambda x: x + pd.Timedelta(hours=1) if start_time <= x <= end_time else x
)

In [25]:
# Define the start and end of the time range
start_time = pd.to_datetime('2022-03-27 01:00:00')
end_time = pd.to_datetime('2022-10-30 02:00:00')

# Apply the condition to add 1 hour only to times within the specified range
kl_divergence_data['time'] = kl_divergence_data['time'].apply(
    lambda x: x + pd.Timedelta(hours=1) if start_time <= x <= end_time else x
)

In [26]:
# Define the start and end of the time range
start_time = pd.to_datetime('2021-03-28 01:00:00')
end_time = pd.to_datetime('2022-10-31 02:00:00')

# Apply the condition to add 1 hour only to times within the specified range
kl_divergence_data['time'] = kl_divergence_data['time'].apply(
    lambda x: x + pd.Timedelta(hours=1) if start_time <= x <= end_time else x
)

## Export Data

In [ ]:
#connect to postgres database
engine = psql.create_engine('postgresql://user:@localhost/db_name')

In [30]:
#send data to sql table
kl_divergence_data.to_sql('kl_divergence_node_data',
                   con=engine,
                   if_exists='append',
                   index=False)

90